<a href="https://colab.research.google.com/github/omoumi/ham10000-skin-classification/blob/main/ham10000_classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch
print("PyTorch version:", torch.__version__)
print("GPU available:", torch.cuda.is_available())
print("GPU name:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")

PyTorch version: 2.11.0+cu128
GPU available: True
GPU name: Tesla T4


In [2]:
from google.colab import files
uploaded = files.upload()

Saving kaggle.json to kaggle.json


In [3]:
import os

os.makedirs("/root/.kaggle", exist_ok=True)
os.rename("kaggle.json", "/root/.kaggle/kaggle.json")
os.chmod("/root/.kaggle/kaggle.json", 0o600)

print("Kaggle key is set up.")

Kaggle key is set up.


In [4]:
!pip install -q kaggle

In [5]:
!kaggle datasets download -d kmader/skin-cancer-mnist-ham10000

Dataset URL: https://www.kaggle.com/datasets/kmader/skin-cancer-mnist-ham10000
License(s): CC-BY-NC-SA-4.0
100% 5.20G/5.20G [00:41<00:00, 135MB/s]



In [6]:
!unzip -q skin-cancer-mnist-ham10000.zip -d ham10000

In [7]:
import os

for item in os.listdir("ham10000"):
    print(item)


hmnist_8_8_RGB.csv
HAM10000_metadata.csv
HAM10000_images_part_2
ham10000_images_part_1
hmnist_28_28_RGB.csv
HAM10000_images_part_1
ham10000_images_part_2
hmnist_8_8_L.csv
hmnist_28_28_L.csv


In [8]:
import pandas as pd

metadata = pd.read_csv("ham10000/HAM10000_metadata.csv")

print("Shape:", metadata.shape)
metadata.head()

Shape: (10015, 7)


,lesion_id,image_id,dx,dx_type,age,sex,localization
0,HAM_0000118,ISIC_0027419,bkl,histo,80.0,male,scalp
1,HAM_0000118,ISIC_0025030,bkl,histo,80.0,male,scalp
2,HAM_0002730,ISIC_0026769,bkl,histo,80.0,male,scalp
3,HAM_0002730,ISIC_0025661,bkl,histo,80.0,male,scalp
4,HAM_0001466,ISIC_0031633,bkl,histo,75.0,male,ear


In [ ]:
print(metadata["dx"].value_counts())

dx
nv       6705
mel      1113
bkl      1099
bcc       514
akiec     327
vasc      142
df        115
Name: count, dtype: int64


In [10]:
labels = sorted(metadata["dx"].unique())
label_to_idx = {name: index for index, name in enumerate(labels)}

metadata["label"] = metadata["dx"].map(label_to_idx)

print(label_to_idx)
metadata[["image_id", "dx", "label"]].head()

{'akiec': 0, 'bcc': 1, 'bkl': 2, 'df': 3, 'mel': 4, 'nv': 5, 'vasc': 6}


,image_id,dx,label
0,ISIC_0027419,bkl,2
1,ISIC_0025030,bkl,2
2,ISIC_0026769,bkl,2
3,ISIC_0025661,bkl,2
4,ISIC_0031633,bkl,2


In [19]:
import glob

image_paths = {}
for folder in ["HAM10000_images_part_1", "HAM10000_images_part_2"]:
  for path in glob.glob(f"ham10000/{folder}/*.jpg"):
    image_id = os.path.basename(path).replace(".jpg", "")
    image_paths[image_id] = path

print("Total images found:", len(image_paths))
print("Example:", list(image_paths.items())[0])

Total images found: 10015
Example: ('ISIC_0028919', 'ham10000/HAM10000_images_part_1/ISIC_0028919.jpg')


In [20]:
from torch.utils.data import Dataset
from PIL import Image

class HAM10000Dataset(Dataset):
  def __init__(self, dataframe, image_paths, transform=None):
    self.dataframe = dataframe.reset_index(drop=True)
    self.image_paths = image_paths
    self.transform = transform

  def __len__(self):
    return len(self.dataframe)

  def __getitem__(self, index):
    row = self.dataframe.iloc[index]
    image_id = row["image_id"]
    label = row["label"]

    path = self.image_paths[image_id]
    image = Image.open(path).convert("RGB")

    if self.transform:
      image = self.transform(image)

    return image, label
print("Dataset class defined.")

Dataset class defined.


In [21]:
from sklearn.model_selection import GroupShuffleSplit

splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
groups = metadata["lesion_id"]

train_idx, test_idx = next(splitter.split(metadata, groups=groups))

train_df = metadata.iloc[train_idx]
test_df = metadata.iloc[test_idx]

print("Train images:", len(train_df))
print("Test images:", len(test_df))

Train images: 7991
Test images: 2024


In [22]:
train_lesions = set(train_df["lesion_id"])
test_lesions = set(test_df["lesion_id"])

overlap = train_lesions.intersection(test_lesions)

print("Unique lesions in train:", len(train_lesions))
print("Unique lesions in test:", len(test_lesions))
print("Overlapping lesions:", len(overlap))

Unique lesions in train: 5976
Unique lesions in test: 1494
Overlapping lesions: 0


In [23]:
import torchvision.transforms as transforms
from torch.utils.data import DataLoader

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225))
])

train_dataset = HAM10000Dataset(train_df, image_paths, transform=transform)
test_dataset = HAM10000Dataset(test_df, image_paths, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

print("Train dataset size:", len(train_dataset))
print("Test dataset size:", len(test_dataset))

Train dataset size: 7991
Test dataset size: 2024


In [24]:
image, label = train_dataset[0]

print("Image type:", type(image))
print("Image shape:", image.shape)
print("Label:", label)

Image type: <class 'torch.Tensor'>
Image shape: torch.Size([3, 224, 224])
Label: 2


In [25]:
import torch.nn as nn
from torchvision import models

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
model.fc = nn.Linear(model.fc.in_features, 7)
model = model.to(device)

print("Model ready. Output classes:", model.fc.out_features)

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 193MB/s]


Model ready. Output classes: 7


In [26]:
import numpy as np
from sklearn.utils.class_weight import compute_class_weight

classes_array = np.array(sorted(train_df["label"].unique()))
weights = compute_class_weight(
    class_weight="balanced",
    classes=classes_array,
    y=train_df["label"]
)

class_weights = torch.tensor(weights, dtype=torch.float).to(device)

print("Class weights:")
for i, w in enumerate(class_weights):
    print(f"  Class {i} ({labels[i]}): {w:.3f}")

Class weights:
  Class 0 (akiec): 4.391
  Class 1 (bcc): 2.692
  Class 2 (bkl): 1.334
  Class 3 (df): 11.769
  Class 4 (mel): 1.302
  Class 5 (nv): 0.213
  Class 6 (vasc): 10.570


In [27]:
import torch.optim as optim

criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = optim.Adam(model.parameters(), lr=0.0001)

print("Loss function is now weighted by class.")

Loss function is now weighted by class.


In [28]:
epochs = 5
loss_history = []

for epoch in range(epochs):
    model.train()
    running_loss = 0.0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    average_loss = running_loss / len(train_loader)
    loss_history.append(average_loss)
    print(f"Epoch {epoch+1}/{epochs} - Loss: {average_loss:.3f}")

Epoch 1/5 - Loss: 1.066
Epoch 2/5 - Loss: 0.462
Epoch 3/5 - Loss: 0.227
Epoch 4/5 - Loss: 0.135
Epoch 5/5 - Loss: 0.070


In [31]:
class_names = [name for name, idx in sorted(label_to_idx.items(), key=lambda x: x[1])]

print("Class names in order:", class_names)
print()
print(classification_report(all_labels, all_preds, target_names=class_names))

Class names in order: ['akiec', 'bcc', 'bkl', 'df', 'mel', 'nv', 'vasc']

              precision    recall  f1-score   support

       akiec       0.43      0.66      0.52        67
         bcc       0.61      0.76      0.67        90
         bkl       0.71      0.67      0.69       243
          df       0.50      0.44      0.47        18
         mel       0.62      0.53      0.57       236
          nv       0.92      0.91      0.91      1336
        vasc       0.83      0.88      0.86        34

    accuracy                           0.82      2024
   macro avg       0.66      0.69      0.67      2024
weighted avg       0.82      0.82      0.82      2024

